# Neurotransmitter Probability Variance across Drosophila Neuropils

This projects aims to answer - how do neurotransmitter probability distributions vary across neuropils in Drosophila?
The datasets should be downloaded into the data directory following the instructions on GitHub. 

## Stage 0: Set up environment

In [1]:
from pyvista.trame import TramePlotter
TramePlotter


ImportError: cannot import name 'TramePlotter' from 'pyvista.trame' (C:\Users\cielb\anaconda3\envs\nta1\Lib\site-packages\pyvista\trame\__init__.py)

In [1]:
%load_ext autoreload 
%autoreload 2

# Import external libraries
import dask
from IPython.core.display import HTML, Image
import pyvista as pv
from dask.distributed import Client

# Import core python libraries
import os

# Import local scripts (brainz.py, pipeline.py, plotting.py, 
# preprocess.py, util.py)
from scripts import *

ImportError: cannot import name 'TramePlotter' from 'pyvista.trame' (C:\Users\cielb\anaconda3\envs\nta1\Lib\site-packages\pyvista\trame\__init__.py)

In [2]:
import pyvista.trame as pvt
dir(pvt)


['PyVistaLocalView',
 'PyVistaRemoteLocalView',
 'PyVistaRemoteView',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 'annotations',
 'elegantly_launch',
 'get_viewer',
 'jupyter',
 'launch_server',
 'logging',
 'plotter_ui',
 'show_trame',
 'ui',
 'views']

In [ ]:
# Set up in-line 3D plot rendering
pv.set_jupyter_backend("client")

In [ ]:
# Set globals
OUTDIR = os.path.join(os.path.dirname(__name__), "results", "notebook")
if not os.path.exists(OUTDIR): os.mkdir(OUTDIR)
MINSIZE = 30 # Minimum number of nodes in a cluster/community

In [ ]:
allow_bytes = pipeline.get_ram_allowance() # Allow Dask to use just over 60% of system RAM
PARTITION_SIZE = pipeline.get_partition_size(num_threads = 4, allow_bytes = allow_bytes)
client = pipeline.start_dask(num_threads = 4, allow_bytes = allow_bytes)
dask.config.set({"dataframe.shuffle.method": "tasks"})
preprocess.run() # ~10 mins first run

## Stage 1: Load Data

This stage of the pipeline loads and merges data from file, creating a single cohesive dataframe containing metadata for every synapse belonging to a neuron-neuron connection in proofread_connections_783.parquet. 

Neurotransmitter probability normalisation is required because the probabilities of some points do not sum to exactly 1 across all 6 neurotransmitters, but Dirichlet regression and composition analysis requires these probabilities to sum to exactly 1. Additionally, neurotransmitter normalisation reduces the data from 6 compositional variables to 3 compositional variables: GABA, an inhibitory neurotransmitter; Acetylcholine ('ach'), an excitatory neurotransmitter, and 'other', which encompasses the remaining neurotransmitters which cannot be reliably classified as solely inhibitory or excitatory (rather, they are usually neuromodulators).

Although the proofread connections file contains a compact representation of synapse data already (synapse count, and the same neurotransmitter probabilities allocated to each synapse), this stage of the pipeline expands the compact representation back to representing one synapse per row. This is because compositional analysis of neurotransmitter probabilities across synapses (e.g., in Dirichlet regression and in plotting ternary plots) requires this expanded format. 

In [ ]:
# Choose dataset to use. Note the decisions made in this notebook are based
# on the full connectome dataset (proofread_connections_783.parquet). The
# other datasets are selections of different neuropils, and are provided for
# debugging/testing purposes.

DATASET = "data/proofread_connections_783.parquet" # ~ 1.05 GB

#DATASET = "data/large.parquet" # ~ 583 MB
#DATASET = "data/medium.parquet" # ~ 272 MB
#DATASET = "data/small.parquet" # ~ 56 MB
#DATASET = "data/tiny.parquet" # ~ 215 KB

In [ ]:
pipeline.relax_memory_limits() # Let Dask use a higher percentage of allocated RAM

In [ ]:
connectome = pipeline.load_connectome(DATASET, PARTITION_SIZE)
connectome.head(3, npartitions=1).style.hide(axis="index")

In [ ]:
connectome = pipeline.normalise_nt_probs(connectome)
connectome.head(3, npartitions=1).style.hide(axis="index")

In [ ]:
# This step takes a little longer because it merges the 1 GB 
# proofread_synapses_783.parquet file with the ~13 GB 
# flywire_synapses_783.parquet file. 
# Took ~2 min with ~8 GB RAM allowance and 4 threads.
connectome = pipeline.attach_synapse_coords(connectome, PARTITION_SIZE)
connectome.head(3, npartitions=1).style.hide(axis="index")

In [ ]:
connectome = pipeline.attach_neuropil_metadata(connectome)
connectome.head(3, npartitions=1).style.hide(axis="index")

In [ ]:
pipeline.restore_memory_limits() # Next stages are heavy on unmanaged memory

## Stage 2: Identify & Visualise Neurotransmitter Clusters

This step identifies frequent cluster compositions and allocates each point to either a frequent cluster or labels it as 'noise'. A brain map is then plotted, with each point's colour corresponding to its allocated neurotransmitter cluster. Although this approach requires extra effort and will be harder to automate in the future, it is superior to the naive approach of labelling each point on a brain map by the neurotransmitter with the highest probability for that edge. This is because the latter approach ignores important compositional information and can lump two very different neurotransmitter ratios into the same group (e.g., a point with 100% acetylcholine probability would be labelled in the same manner as a point with 34% acetylcholine, 33% gaba, and 33% other probabilities).

### Visualise Initial Neurotransmitter Probability Distribution

In [ ]:
connectome_nts = connectome[["gaba", "ach", "other"]]

In [ ]:
# Sample connectome - hexbin plotting is memory-intensive and don't need every data point to get idea of distribution
sample = pipeline.downsample(connectome_nts, 50_000, allow_bytes)
filename = os.path.join(OUTDIR, "init_overall_distribution.png")
pipeline.make_plot("plot_n_ternary", filename, samples = [sample], labels = ["Neurotransmitter_Probabilities_In_The_Drosophila_Connectome"])
Image(filename, width = 600)

### Divide synapses into 5 different categories

I will divide the synapses into 5 different categories based on the above ternary plot. The bottom left corner will be manually split into two different clusters as it is unclear whether the sheer number of synapses allocated to the yellow hex bin will be part of similar structural features as the surrounding synapses. This unclarity stems partly from the fact the hex bins surrounding the yellow hex bin are a similar shade to the other two clusters at the top corner and bottom right corner. Thus the synapses allocated to the bottom left corner will be divided between those belonging to the yellow hex bin and those belonging to the surrounding clusters. It is possible to merge these clusters again if required. The fifth cluster is not so much a 'cluster', but simply the 'noise' synapses that could not be allocated to any other cluster. 

While the bottom left corner is fairly straightforward to extract, it is difficult to judge exactly where I should set cluster boundaries for the other two corners. Results *may* differ noticeably depending on the exact choice of value. I am only testing one possible clustering.

In [ ]:
# Extract and visualise the four initial clusters
connectome = pipeline.tag_clusters(connectome_nts)

In [ ]:
# Plot hex plots of new cluster classifications
sample = pipeline.downsample(connectome, 50_000, allow_bytes)
grouped = sample.groupby("cluster")
high_gaba_sample = grouped.getgroup("gaba")
high_ach_sample = grouped.getgroup("ach")
high_other_sample = grouped.getgroup("other")
unclassified_sample = grouped.getgroup("none")

filename = os.path.join(OUTDIR, "initial_clusters.png")
pipeline.make_plot("plot_n_ternary", filename, 
                   samples = [high_gaba_sample, high_ach_sample, high_other_sample, unclassified_sample], 
                   labels = ["High_GABA", "High_ACH", "High_OTHER", "Unclassified"])
Image(filename, width = 600)

In [ ]:
# Decide on cluster boundaries for high acetylcholine synapses
...

In [ ]:
filename = os.path.join(OUTDIR, "final_clusters.png")
pipeline.make_plot("plot_n_ternary", filename, 
                   samples = [high_gaba_sample, high_ach_sample, high_other_sample, unclassified_sample], 
                   labels = ["High_GABA", "High_ACH", "High_OTHER", "Unclassified"])
Image(filename)

### Generate Brain Map

In [ ]:
# Get sample of clustered synapses
sample = 

In [ ]:
# Plot brain map with 500,000 points
plotter = brainz.get_plotter(clustered, "hdbscan_id")
brainz.save(plotter, OUTDIR, _id="clusterd_brain_map")
print("Saved brain map!")
plotter.show()

In [ ]:
plotter.close() # Free GPU memory

## Stage 3: Visualise Neurotransmitter Probability Distributions Across Neuropils

### Mean and Variance of Neurotransmitter Probabilities Across Various Neuropil Sizes

### Neurotransmitter Probabilities Across Different Brain Regions

## Stage 4: Statistical Analysis

I want to know whether neurotransmitter probabilities are different between neuropils. Because probabilities are a form of compositional data (the sum of the variables is 1, and each variable is bounded between 0 and 1), the Dirichlet distribution is suitable. This statistical analysis is a simple one based on average synapse probabilities within neuropils. A more statistically robust method would involve using the 'other' column to calculate the ranges of excitatory and inhibitory probabilities for each synaptic connection, then running a statistical analysis that can handle comparing range values within a bounded interval, e.g., Bayesian Dirichlet regression with interval priors. The Dirichlet function I used can only operate on points, not ranges. There does not appear to be an out-of-box Dirichlet regression function in any Python libraries, so I will interface with the R DirichletReg package via rpy2.

### Get Neuropil Summary Statistics

In [ ]:
neuropils = pipeline.get_neuropil_summary_stats(connectome)
neuropils.head(10)

### Run statistical_analysis.R